In [ ]:
#data_utils.py functions
import re
from functools import lru_cache
import numba

def create_word_column_verbose(df, feature_columns):
    """
    创建 'word' 列的详细版本，会打印出每一步的信息，用于调试。
    """
    print(f"Input DataFrame shape: {df.shape}")
    print(f"Requested feature columns: {feature_columns}")

    # Check which columns exist
    existing_cols = [col for col in feature_columns if col in df.columns]
    missing_cols = [col for col in feature_columns if col not in df.columns]

    print(f"Existing columns: {existing_cols}")
    print(f"Missing columns: {missing_cols}")

    if not existing_cols:
        print("No requested columns found in DataFrame!")
        print(f"Available columns: {df.columns.tolist()}")
        # Create placeholder
        df['word'] = 'placeholder'
        return df

    # Show sample data from each column
    for col in existing_cols:
        try:
            print(f"Column '{col}' - types: {df[col].dtype}, sample values: {df[col].head(3).tolist()}")
        except Exception as e:
            print(f"Error inspecting column '{col}': {e}")

    # Create the word column
    try:
        string_values = []
        for i, row in df.iloc[:100].iterrows():  # Process only first 100 rows for inspection
            parts = []
            for col in existing_cols:
                if pd.notna(row[col]):
                    parts.append(str(row[col]))
                else:
                    parts.append("0")
            string_values.append(' '.join(parts))

        print(f"Sample generated words (first 3): {string_values[:3]}")

        # Now process the entire DataFrame
        df['word'] = df[existing_cols].astype(str).agg(' '.join, axis=1)
        print(f"Word column created successfully. Sample: {df['word'].head(3).tolist()}")

    except Exception as e:
        print(f"Error creating word column: {e}")
        # Fallback to row-by-row processing
        print("Falling back to slower row-by-row processing...")
        string_values = []
        for i, row in df.iterrows():
            parts = []
            for col in existing_cols:
                if pd.notna(row[col]):
                    parts.append(str(row[col]))
                else:
                    parts.append("0")
            string_values.append(' '.join(parts))

        df['word'] = string_values

    return df

# Global mappings (unchanged)
ENHARMONIC_MAPPING = {
    # 双降音符
    'Cbb': 'Bb',
    'Dbb': 'C',
    'Ebb': 'D',
    'Fbb': 'Eb',
    'Gbb': 'F',
    'Abb': 'G',
    'Bbb': 'A',

    # 单降音符
    'Cb': 'B',
    'Db': 'C#',
    'Eb': 'D#',
    'Fb': 'E',
    'Gb': 'F#',
    'Ab': 'G#',
    'Bb': 'A#',

    # 纯音符
    'C': 'C',
    'D': 'D',
    'E': 'E',
    'F': 'F',
    'G': 'G',
    'A': 'A',
    'B': 'B',

    # 单升音符
    'C#': 'C#',
    'D#': 'D#',
    'E#': 'F',
    'F#': 'F#',
    'G#': 'G#',
    'A#': 'A#',
    'B#': 'C',

    # 双升音符
    'Cx': 'D',
    'Dx': 'E',
    'Ex': 'F#',
    'Fx': 'G',
    'Gx': 'A',
    'Ax': 'B',
    'Bx': 'C#',
}

TEMPO_MAPPING = {
    'Larghissimo': 24,
    'Grave': 40,
    'Largo': 40,
    'Larghetto': 50,
    'Adagio': 66,
    'Adagietto': 68,
    'Andante': 76,
    'Andantino': 80,
    'Moderato': 108,
    'Allegretto': 112,
    'Allegro': 120,
    'Vivace': 156,
    'Presto': 168,
    'Prestissimo': 200
}

# Pre-compute the reverse mapping
REVERSE_ENHARMONIC_MAPPING = {}
for key, value in ENHARMONIC_MAPPING.items():
    if value not in REVERSE_ENHARMONIC_MAPPING:
        REVERSE_ENHARMONIC_MAPPING[value] = []
    REVERSE_ENHARMONIC_MAPPING[value].append(key)

# Pre-compile regular expressions for better performance
PITCH_REGEX = re.compile(r'^([A-Ga-g][#b]{0,2})(\d+)$')


class CustomFingeringEncoder:
    """自定义指法编码器，处理钢琴指法的编码和解码。
       右手指法为正数，左手指法为负数；新数据集中未标注的指法（0）将被映射为10，
       并在训练时通过损失函数忽略（ignore_index=10）。
    """

    def __init__(self):
        self.mapping = {
            -5: 0, -4: 1, -3: 2, -2: 3, -1: 4,  # 左手指法：-5到-1映射到0-4
            0: 10,  # 未标注的指法（0）映射为10（ignore index）
            1: 5, 2: 6, 3: 7, 4: 8, 5: 9  # 右手指法：1到5映射到5-9
        }
        self.inverse_mapping = {v: k for k, v in self.mapping.items()}

    def fit(self, y):
        # 验证数据中的指法是否都在映射范围内
        invalid_fingers = set(y) - set(self.mapping.keys())
        if invalid_fingers:
            raise ValueError(f"Found invalid fingerings: {invalid_fingers}")
        return self

    def transform(self, y):
        # Use numpy vectorize for better performance
        transform_func = np.vectorize(lambda x: self.mapping.get(x, -1))
        return transform_func(y)

    def inverse_transform(self, y):
        # Use numpy vectorize for better performance
        inverse_func = np.vectorize(lambda x: self.inverse_mapping.get(x, 0))
        return inverse_func(y)

    def __getstate__(self):
        return {'mapping': self.mapping, 'inverse_mapping': self.inverse_mapping}

    def __setstate__(self, state):
        self.mapping = state['mapping']
        self.inverse_mapping = state['inverse_mapping']


# Use LRU cache to avoid repeated calculations for common pitches
@lru_cache(maxsize=128)
def normalize_spelled_pitch(spelled_pitch):
    """标准化音高表示"""
    try:
        match = PITCH_REGEX.match(spelled_pitch)
        if not match:
            return "C4"  # Default value for invalid format
        pitch_name, octave = match.groups()
        normalized_pitch = ENHARMONIC_MAPPING.get(pitch_name, pitch_name)
        return f"{normalized_pitch}{octave}"
    except Exception:
        return "C4"  # Default value for any exception


# Use LRU cache to avoid repeated calculations
@lru_cache(maxsize=128)
def denormalize_spelled_pitch(normalized_pitch):
    """
    根据需要将标准化后的音符还原为原始的同音异名形式。
    """
    pitch_name = ''.join([c for c in normalized_pitch if c.isalpha() or c in ['#', 'b']])
    octave = ''.join([c for c in normalized_pitch if c.isdigit()])
    original_pitch = REVERSE_ENHARMONIC_MAPPING.get(pitch_name, [pitch_name])[0]  # 默认选第一个
    return f"{original_pitch}{octave}"


# Pre-compute note to semitone mapping
NOTE_TO_SEMITONE = {
    'C': 0, 'C#': 1, 'Db': 1,
    'D': 2, 'D#': 3, 'Eb': 3,
    'E': 4, 'Fb': 4, 'E#': 5,
    'F': 5, 'F#': 6, 'Gb': 6,
    'G': 7, 'G#': 8, 'Ab': 8,
    'A': 9, 'A#': 10, 'Bb': 10,
    'B': 11, 'Cb': 11, 'B#': 0
}


# Use LRU cache and regex for better performance
@lru_cache(maxsize=256)
def get_midi_number(spelled_pitch):
    """
    将标准化后的 spelled_pitch 转换为 MIDI 编号。
    """
    match = PITCH_REGEX.match(spelled_pitch)
    if not match:
        return 60  # Default C4

    pitch, octave = match.groups()
    octave = int(octave)
    semitone = NOTE_TO_SEMITONE.get(pitch, 0)
    midi_number = 12 * (octave + 1) + semitone
    return midi_number


# Use a set for faster lookups
BLACK_KEY_SET = {1, 3, 6, 8, 10}  # C#, D#, F#, G#, A#


def is_black_key(midi_number):
    """
    判断MIDI编号对应的音符是否为黑键。
    """
    return 1 if (midi_number % 12) in BLACK_KEY_SET else 0


# Use Numba to accelerate this computation-heavy function
@numba.jit(nopython=True)
def calculate_density_numba(onset_times, window=1.0):
    """
    Numba-accelerated version of calculate_density
    """
    n = len(onset_times)
    sorted_onsets = np.sort(onset_times)
    density = np.zeros(n, dtype=np.int32)

    for i in range(n):
        t = onset_times[i]
        # Count notes falling within the window
        count = 0
        for j in range(n):
            if sorted_onsets[j] >= t and sorted_onsets[j] <= t + window:
                count += 1
        density[i] = count

    return density


def calculate_density(df, window=1.0):
    """
    使用向量化的方法计算每个音符在给定时间窗口内的密度。
    """
    # 提取所有音符的 onset_time 并排序
    onset_times = df['onset_time'].values

    # Try to use the Numba-accelerated version when possible
    try:
        return calculate_density_numba(onset_times, window)
    except:
        # Fallback to numpy version
        sorted_onsets = np.sort(onset_times)
        lower_indices = np.searchsorted(sorted_onsets, onset_times, side='left')
        upper_indices = np.searchsorted(sorted_onsets, onset_times + window, side='right')
        return (upper_indices - lower_indices).tolist()


def calculate_speed_features(df, window=1.0):
    """计算速度相关特征"""
    df = df.copy()

    # 计算真实时值
    df['real_duration'] = df['offset_time'] - df['onset_time']

    # 计算稠密度（向量化实现）
    df['note_density'] = calculate_density(df, window)
    return df


def calculate_midi_diff(df):
    """
    计算 MIDI 差值和处理后的 MIDI 差值。
    """
    df = df.copy()

    # Use shift with efficient filling
    df['prev_midi_number'] = df['midi_number'].shift(1, fill_value=60)
    df['midi_diff'] = df['midi_number'] - df['prev_midi_number']

    # Add chord indicator if missing
    if 'chord' not in df.columns:
        df['chord'] = 0  # Default not a chord

    # Vectorize the processing of midi_diff
    def process_midi_diff(row):
        if row['chord']:
            # Based on formula (3), assuming k=0
            return 200 * 0 - row['midi_diff']
        else:
            if row['midi_diff'] < 100:
                return -row['midi_diff']
            else:
                return row['midi_diff']

    # Apply in chunks to save memory
    chunk_size = 100000
    result = []

    for i in range(0, len(df), chunk_size):
        chunk = df.iloc[i:i + chunk_size]
        processed = chunk.apply(process_midi_diff, axis=1)
        result.append(processed)

    df['midi_diff_processed'] = pd.concat(result)
    return df


def create_word_column(df, feature_columns):
    """
    创建 'word' 列，将多个特征组合成一个字符串，用于Word2Vec训练。
    """
    # Make a copy to avoid SettingWithCopyWarning
    df = df.copy()

    # Verify all columns exist, use only available columns
    available_cols = [col for col in feature_columns if col in df.columns]
    if not available_cols:
        print(f"Warning: None of the requested feature columns {feature_columns} found in DataFrame.")
        # Create a default word column with a placeholder
        df['word'] = 'placeholder'
        return df

    # Convert to string column by column and join to save memory
    string_values = []
    for i, row in df.iterrows():
        parts = []
        for col in available_cols:
            if pd.notna(row[col]):  # Handle NaN values
                parts.append(str(row[col]))
            else:
                parts.append("0")  # Use "0" as placeholder for missing values
        string_values.append(' '.join(parts))

    df['word'] = string_values
    return df


def train_word2vec(sentences, window=5, vector_size=64, min_count=5, workers=4):
    """训练Word2Vec模型，使用CBOW"""
    model = Word2Vec(sentences, window=window, vector_size=vector_size,
                     min_count=min_count, workers=workers, sg=0)  # sg=0表示CBOW
    return model


def get_fused_features(df, word2vec_model, tokenized_sentences):
    """
    获取融合特征，通过 Word2Vec 将 'word' 列转换为向量，确保最终向量维度为128。
    对于未在词汇表中的单词，使用全零向量代替。
    优化点：使用局部变量缓存词向量对象，减少函数调用开销。
    内存优化版本：按更小批次处理，使用float32减小内存占用。
    """
    wv = word2vec_model.wv
    vector_size = wv.vector_size

    # Process in very small batches to avoid memory issues
    batch_size = min(500, len(df))
    all_vectors = []

    # Force garbage collection before starting
    gc.collect()

    for i in range(0, len(tokenized_sentences), batch_size):
        # Free memory before processing
        gc.collect()

        end = min(i + batch_size, len(tokenized_sentences))
        batch_tokens = tokenized_sentences[i:end]

        # Process each sample in the batch
        batch_vectors = []
        for tokens in batch_tokens:
            # Get vectors for tokens in vocabulary
            token_vectors = []
            for token in tokens:
                if token in wv:
                    # Get vector and immediately convert to float32
                    token_vectors.append(wv[token].astype(np.float32))

            # Calculate mean vector or use zeros
            if token_vectors:
                # Use float32 to reduce memory usage
                base_vector = np.mean(token_vectors, axis=0).astype(np.float32)
            else:
                base_vector = np.zeros(vector_size, dtype=np.float32)

            # Ensure 128-dimensional output
            if base_vector.shape[0] > 128:
                result_vector = base_vector[:128]
            elif base_vector.shape[0] < 128:
                padding = np.zeros(128 - base_vector.shape[0], dtype=np.float32)
                result_vector = np.concatenate([base_vector, padding])
            else:
                result_vector = base_vector

            batch_vectors.append(result_vector)

            # Free memory for token vectors
            del token_vectors

        all_vectors.extend(batch_vectors)

        # Clean up to free memory
        del batch_vectors, batch_tokens
        gc.collect()

    # Assign vectors to DataFrame
    df['fused_feature'] = all_vectors

    # Verify dimension
    sample_dim = len(df['fused_feature'].iloc[0])
    print(f"Fused feature dimension before scaling: {sample_dim}")
    assert sample_dim == 128, f"Expected 128 features for fusion, got {sample_dim}"

    return df


def combine_features(df, feature_columns):
    """将融合特征与原始特征组合，确保总维度为136（基础特征8维 + 融合特征128维）。
    内存优化版本：按小批次处理，使用低精度数据类型。
    """
    # Process in very small batches to save memory
    batch_size = min(1000, len(df))
    all_combined = []

    # Force garbage collection before starting
    gc.collect()

    for i in range(0, len(df), batch_size):
        # Free memory before processing each batch
        gc.collect()

        batch = df.iloc[i:i + batch_size]

        # Get base features (ensure all columns exist)
        valid_columns = [col for col in feature_columns if col in batch.columns]

        # Convert to float32 to reduce memory usage
        base_features = batch[valid_columns].values.astype(np.float32)

        # If we have fewer than 8 base features, pad with zeros
        if base_features.shape[1] < 8:
            padding = np.zeros((base_features.shape[0], 8 - base_features.shape[1]), dtype=np.float32)
            base_features = np.concatenate([base_features, padding], axis=1)

        # Get fused features and convert to float32
        fused_features = np.stack(batch['fused_feature_scaled'].values).astype(np.float32)

        # Combine with lower precision
        combined = np.concatenate([base_features, fused_features], axis=1)

        # Store as list
        all_combined.extend(list(combined))

        # Clean up to free memory
        del base_features, fused_features, combined
        gc.collect()

    # Store combined features
    gc.collect()  # One more collection before assignment
    df['combined_features'] = all_combined

    # Verify dimension
    sample_dim = len(df['combined_features'].iloc[0])
    print(f"Combined feature dimension: {sample_dim}")
    assert sample_dim == 136, f"Expected 136 features, got {sample_dim}"

    return df


def process_fused_features(df, scaler_fused=None, batch_size=10000):
    """
    将融合特征分批处理，并使用增量标准化。
    如果 scaler_fused 为 None，则新建一个 StandardScaler，否则使用传入的 scaler 进行增量拟合。
    返回标准化后的融合特征列表。
    内存优化版本：使用更小的批次和float32数据类型。
    """
    # Verify the 'fused_feature' column exists
    if 'fused_feature' not in df.columns:
        raise ValueError("Column 'fused_feature' not found in DataFrame. Make sure to run get_fused_features() first.")

    # Create scaler if needed
    if scaler_fused is None:
        scaler_fused = StandardScaler()

    # Process in smaller chunks to save memory
    chunk_size = min(batch_size, len(df))
    all_scaled = []

    # Force garbage collection before starting
    gc.collect()

    for i in range(0, len(df), chunk_size):
        # Free memory before processing
        gc.collect()

        # Extract chunk of data
        chunk = df.iloc[i:i + chunk_size]

        # Convert list to array with lower precision
        fused_features_array = np.array(chunk['fused_feature'].tolist(), dtype=np.float32)

        # Fit or partial_fit the scaler
        if i == 0 and not hasattr(scaler_fused, 'mean_'):
            scaler_fused.fit(fused_features_array)
        else:
            scaler_fused.partial_fit(fused_features_array)

        # Transform and store with lower precision
        scaled_chunk = scaler_fused.transform(fused_features_array).astype(np.float32)
        all_scaled.append(scaled_chunk)

        # Free memory
        del fused_features_array, scaled_chunk
        gc.collect()

    # Combine all scaled chunks
    if len(all_scaled) > 1:
        # Combine chunks in smaller groups to avoid memory issues
        final_scaled = []
        group_size = 5  # Combine at most 5 chunks at a time

        for j in range(0, len(all_scaled), group_size):
            end_j = min(j + group_size, len(all_scaled))
            if end_j - j > 1:
                group = np.vstack(all_scaled[j:end_j])
            else:
                group = all_scaled[j]
            final_scaled.append(group)

            # Clean up to free memory
            for k in range(j, end_j):
                all_scaled[k] = None
            gc.collect()

        if len(final_scaled) > 1:
            fused_features_scaled = np.vstack(final_scaled)
        else:
            fused_features_scaled = final_scaled[0]
    else:
        fused_features_scaled = all_scaled[0]

    # Clean up
    del all_scaled
    gc.collect()

    return scaler_fused, fused_features_scaled


def save_pickle(obj, filename):
    """
    保存对象为pickle文件。
    """
    with open(filename, 'wb') as f:
        pickle.dump(obj, f)


def load_pickle(filename):
    """
    从pickle文件加载对象。
    """
    with open(filename, 'rb') as f:
        return pickle.load(f)

In [3]:
#dataset_prep
import pickle
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from multiprocessing import Pool
from data_utils import (
    get_midi_number,
    save_pickle,
    normalize_spelled_pitch,
    ENHARMONIC_MAPPING,
    REVERSE_ENHARMONIC_MAPPING,
    CustomFingeringEncoder
)

def parse_fingering_file(file_path):
    """解析单个指法文件，返回结构化数据"""
    piece_id = os.path.splitext(os.path.basename(file_path))[0]
    data = []
    with open(file_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 8:
                continue
            note_id = parts[0]
            onset_time = float(parts[1])
            offset_time = float(parts[2])
            spelled_pitch = parts[3]
            onset_velocity = float(parts[4])
            offset_velocity = float(parts[5])
            channel = int(parts[6])
            finger_number = parts[7]

            # 处理指法中的复杂情况（如 "1_2"）
            if '_' in finger_number:
                finger_number = finger_number.split('_')[0]
            try:
                finger_number = int(finger_number)
            except ValueError:
                finger_number = 0

            # 提取音高和八度
            pitch_name = ''.join([c for c in spelled_pitch if c.isalpha() or c in ['#', 'b']])
            octave = ''.join([c for c in spelled_pitch if c.isdigit()])
            octave = int(octave) if octave else 4

            # 判断左右手
            hand = 'right' if channel == 0 else 'left'
            duration = round(offset_time - onset_time, 2)
            normalized_spelled_pitch = normalize_spelled_pitch(spelled_pitch)

            data.append({
                'piece_id': piece_id,
                'note_id': note_id,
                'onset_time': onset_time,
                'offset_time': offset_time,
                'spelled_pitch': spelled_pitch,
                'normalized_spelled_pitch': normalized_spelled_pitch,
                'pitch_name': pitch_name,
                'octave': octave,
                'duration': duration,
                'hand': hand,
                'finger_number': finger_number
            })
    return data

def load_pig_dataset(fingering_dir):
    """多线程加载所有指法文件"""
    files = [os.path.join(fingering_dir, f) for f in os.listdir(fingering_dir) if f.endswith('.txt')]
    with Pool(processes=4) as pool:
        all_data = pool.map(parse_fingering_file, files)
    all_data = [item for sublist in all_data for item in sublist]
    return pd.DataFrame(all_data)

def main():
    # 文件路径
    fingering_folder = 'ThumbSet/FingeringFiles'
    df = load_pig_dataset(fingering_folder)

    # 数据清洗
    df['finger_number'] = df['finger_number'].fillna(0).astype(int)

    # 浮点数精度控制，减少存储空间
    float_columns = ['onset_time', 'offset_time', 'duration']
    df[float_columns] = df[float_columns].round(2)

    # 计算MIDI音高
    df['midi_number'] = df['normalized_spelled_pitch'].apply(get_midi_number)

    # 特征编码
    le_pitch = LabelEncoder()
    le_duration = LabelEncoder()
    le_hand = LabelEncoder()
    fingering_encoder = CustomFingeringEncoder()

    df['pitch_encoded'] = le_pitch.fit_transform(df['normalized_spelled_pitch'])
    df['duration_encoded'] = le_duration.fit_transform(df['duration'].astype(str))
    df['hand_encoded'] = le_hand.fit_transform(df['hand'])
    fingering_encoder.fit(df['finger_number'])
    df['fingering_encoded'] = fingering_encoder.transform(df['finger_number'])

    # 选择关键特征，减少冗余
    X = df[['pitch_encoded', 'duration_encoded', 'hand_encoded', 'midi_number']].values
    y = df['fingering_encoded'].values

    # 划分训练集和验证集，并记录索引
    train_idx, val_idx = train_test_split(
        df.index, test_size=0.2, random_state=42, stratify=y
    )

    # 在 df 中添加 'train' 列，1 表示训练集，0 表示验证集
    df['train'] = 0
    df.loc[train_idx, 'train'] = 1

    # 根据索引提取训练集和验证集
    X_train = X[train_idx]
    y_train = y[train_idx]
    X_val = X[val_idx]
    y_val = y[val_idx]

    # 生成序列
    sequence_length = 10

    def create_sequences(X, y, seq_length):
        X_seq = []
        y_seq = []
        for i in range(len(X) - seq_length):
            X_seq.append(X[i:i + seq_length])
            y_seq.append(y[i + seq_length])
        return np.array(X_seq, dtype=np.float32), np.array(y_seq, dtype=np.int64)

    X_train_seq, y_train_seq = create_sequences(X_train, y_train, sequence_length)
    X_val_seq, y_val_seq = create_sequences(X_val, y_val, sequence_length)

    # 保存数据
    np.save('X_train.npy', X_train_seq)
    np.save('X_val.npy', X_val_seq)
    np.save('y_train.npy', y_train_seq)
    np.save('y_val.npy', y_val_seq)

    # 保存编码器和数据
    save_pickle(le_pitch, 'le_pitch.pkl')
    save_pickle(le_duration, 'le_duration.pkl')
    save_pickle(le_hand, 'le_hand.pkl')
    save_pickle(fingering_encoder, 'le_fingering.pkl')
    save_pickle(df, 'df.pkl')  # 现在 df 包含 'train' 列
    save_pickle(ENHARMONIC_MAPPING, 'enharmonic_mapping.pkl')
    save_pickle(REVERSE_ENHARMONIC_MAPPING, 'reverse_enharmonic_mapping.pkl')

if __name__ == "__main__":
    main()

done


In [7]:
from data_utils import (
    get_midi_number,
    is_black_key,
    calculate_speed_features,
    calculate_midi_diff,
    create_word_column,
    train_word2vec,
    get_fused_features,
    combine_features,
    save_pickle,
    load_pickle,
    process_fused_features,
    create_word_column_verbose
)
from gensim.models import Word2Vec
from multiprocessing import cpu_count
import numpy as np
import pandas as pd
import torch
import gc
import psutil
import os
import warnings
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler

def combine_npy_files_safely(file_list, output_file, axis=0):
    """
    安全地合并多个npy文件到一个大文件，使用内存映射方式避免OOM

    Args:
        file_list: 要合并的npy文件列表
        output_file: 输出的文件名
        axis: 合并的轴，默认为0
    """
    # First determine the shape and dtype by loading the first file
    print(f"Analyzing files for shape and dtype information...")
    sample_array = np.load(file_list[0])
    dtype = sample_array.dtype

    # Get shape of the entire array
    combined_shape = list(sample_array.shape)

    # Load all arrays to get the total size along the specified axis
    total_size = sample_array.shape[axis]
    for file in tqdm(file_list[1:]):
        # Just load the file to get its shape, without storing entire arrays
        arr = np.load(file)
        total_size += arr.shape[axis]

        # Verify compatible shapes for other dimensions
        for dim in range(len(arr.shape)):
            if dim != axis and arr.shape[dim] != combined_shape[dim]:
                raise ValueError(f"Arrays have incompatible shapes: {arr.shape} vs {combined_shape}")

    # Update the combined shape
    combined_shape[axis] = total_size

    print(f"Creating combined array with shape {combined_shape}, dtype {dtype}")

    # Create memory-mapped output file
    fp = np.lib.format.open_memmap(output_file, mode='w+', dtype=dtype, shape=tuple(combined_shape))

    # Write arrays to the memory-mapped file
    pos = 0
    for i, file in enumerate(tqdm(file_list)):
        if i % 10 == 0:
            gc.collect()  # Periodic GC

        # Load chunk
        chunk = np.load(file)

        # Write to the appropriate position
        if axis == 0:
            fp[pos:pos + chunk.shape[0]] = chunk
            pos += chunk.shape[0]
        else:
            # For other axes, need different slicing
            idx = [slice(None)] * len(combined_shape)
            idx[axis] = slice(pos, pos + chunk.shape[axis])
            fp[tuple(idx)] = chunk
            pos += chunk.shape[axis]

        # Force write to disk and free memory
        fp.flush()
        del chunk
        gc.collect()

    # Close the memmap file
    del fp
    gc.collect()

    print(f"Successfully combined arrays to {output_file}")
    return output_file

# Ignore specific warnings that might clutter output
warnings.filterwarnings('ignore', category=UserWarning, module='gensim')
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message='.*DataFrame.applymap.*')


# Helper function to report memory usage
def report_memory():
    process = psutil.Process(os.getpid())
    memory_info = process.memory_info()
    memory_mb = memory_info.rss / (1024 * 1024)
    print(f"Memory usage: {memory_mb:.2f} MB")


def process_batch(df_batch, word2vec_model, feature_columns, batch_tokenized_sentences):
    """分批处理特征工程"""
    # Create a copy to avoid SettingWithCopyWarning
    df_batch = df_batch.copy()

    # Add music features
    df_batch = calculate_midi_diff(df_batch)
    df_batch = calculate_speed_features(df_batch)
    df_batch['black_key'] = df_batch['midi_number'].apply(is_black_key)
    if 'is_chord' not in df_batch.columns:
        df_batch['is_chord'] = 0
    df_batch['chord'] = df_batch['is_chord']

    # Create word column
    df_batch = create_word_column(df_batch, feature_columns)

    # Get tokenized sentences
    if batch_tokenized_sentences is None:
        batch_tokenized_sentences = df_batch['word'].apply(lambda x: x.split()).tolist()

    # Get features
    df_batch = get_fused_features(df_batch, word2vec_model, batch_tokenized_sentences)

    # Free memory
    gc.collect()

    return df_batch


def sequence_generator(X, y, seq_length, batch_size):
    """生成器，按需生成序列"""
    n_samples = len(X) - seq_length
    for i in range(0, n_samples, batch_size):
        end = min(i + batch_size, n_samples)
        X_batch = np.array([X[j:j + seq_length] for j in range(i, end)], dtype=np.float32)
        y_batch = np.array([y[j + seq_length] for j in range(i, end)], dtype=np.int64)
        yield X_batch, y_batch


def process_dataframe_in_batches(df, word2vec_model, feature_columns, batch_size=10000):
    """Process large dataframes in manageable batches"""
    num_batches = (len(df) + batch_size - 1) // batch_size
    processed_dfs = []

    print(f"Processing dataframe in {num_batches} batches")

    for i in tqdm(range(num_batches)):
        start_idx = i * batch_size
        end_idx = min((i + 1) * batch_size, len(df))

        # Get the batch
        df_batch = df.iloc[start_idx:end_idx].copy()

        # Process features for this batch
        df_batch = calculate_midi_diff(df_batch)
        df_batch = calculate_speed_features(df_batch)
        df_batch['black_key'] = df_batch['midi_number'].apply(is_black_key)
        if 'is_chord' not in df_batch.columns:
            df_batch['is_chord'] = 0
        df_batch['chord'] = df_batch['is_chord']

        # Create word column and tokenize
        df_batch = create_word_column(df_batch, feature_columns)
        batch_tokenized_sentences = df_batch['word'].apply(lambda x: x.split()).tolist()

        # Get fused features
        df_batch = get_fused_features(df_batch, word2vec_model, batch_tokenized_sentences)

        processed_dfs.append(df_batch)

        # Force garbage collection to free memory
        gc.collect()

    return pd.concat(processed_dfs, ignore_index=True)


def main(sample_mode=False, sample_size=50000):
    """
    Main function for data processing

    Args:
        sample_mode: If True, process only a subset of data for testing
        sample_size: Size of the sample to use when sample_mode is True
    """
    print("Starting data processing pipeline...")
    report_memory()

    # Load initial data
    try:
        print("Loading base data...")
        df = pd.read_pickle('df.pkl')
        print(f"DataFrame loaded with shape: {df.shape}")
        report_memory()
    except FileNotFoundError as e:
        print(f"Error loading DataFrame: {e}")
        return

    if 'train' not in df.columns:
        print("Error: 'train' column not found in df.")
        return

    # Use only a sample if in sample mode
    if sample_mode:
        print(f"SAMPLE MODE: Using only {sample_size} samples for testing")
        # Take a stratified sample
        n_train = int(sample_size * 0.8)
        n_val = sample_size - n_train

        train_idx = df[df['train'] == 1].sample(n_train, random_state=42).index
        val_idx = df[df['train'] == 0].sample(n_val, random_state=42).index

        # Create a new DataFrame with only the samples
        df = pd.concat([df.loc[train_idx], df.loc[val_idx]])

        # Reset the train column
        df.loc[train_idx, 'train'] = 1
        df.loc[val_idx, 'train'] = 0

        print(f"Sample created with {len(df)} rows")

    # Split into train and validation sets
    print("Splitting into train and validation sets...")
    df_train = df[df['train'] == 1].copy()
    df_val = df[df['train'] == 0].copy()
    print(f"Number of training samples: {len(df_train)}")
    print(f"Number of validation samples: {len(df_val)}")

    # Free memory
    del df
    gc.collect()
    report_memory()

    # Define feature columns
    feature_columns = ['pitch_encoded', 'duration_encoded', 'hand_encoded',
                       'midi_diff_processed', 'real_duration',
                       'note_density', 'black_key', 'chord']

    # Print all columns in the dataframe for debugging
    print(f"\nAll columns in df_train: {df_train.columns.tolist()}")

    # Define batch size based on available memory
    batch_size = min(50000, len(df_train) // 20)  # At most 5% of data per batch
    print(f"Using batch size: {batch_size}")

    # Process training data in batches
    print("Processing training data in batches...")

    # First, add all necessary columns
    train_batches = []
    num_train_batches = (len(df_train) + batch_size - 1) // batch_size

    for i in tqdm(range(num_train_batches)):
        start_idx = i * batch_size
        end_idx = min((i + 1) * batch_size, len(df_train))

        batch = df_train.iloc[start_idx:end_idx].copy()

        # Calculate music features
        batch = calculate_midi_diff(batch)
        batch = calculate_speed_features(batch)
        batch['black_key'] = batch['midi_number'].apply(is_black_key)
        if 'is_chord' not in batch.columns:
            batch['is_chord'] = 0
        batch['chord'] = batch['is_chord']

        train_batches.append(batch)

        # Free memory
        gc.collect()

    # Concatenate processed batches
    df_train = pd.concat(train_batches, ignore_index=True)
    del train_batches
    gc.collect()
    report_memory()

    # Now optimize data types AFTER creating all columns
    float_cols = ['midi_diff_processed', 'real_duration', 'note_density']
    int_cols = ['pitch_encoded', 'duration_encoded', 'hand_encoded', 'black_key', 'chord']

    for col in float_cols:
        if col in df_train.columns:
            df_train[col] = df_train[col].astype('float32')

    for col in int_cols:
        if col in df_train.columns:
            df_train[col] = df_train[col].astype('int16')

    report_memory()

    # Process validation data similarly
    print("Processing validation data in batches...")
    val_batches = []
    num_val_batches = (len(df_val) + batch_size - 1) // batch_size

    for i in tqdm(range(num_val_batches)):
        start_idx = i * batch_size
        end_idx = min((i + 1) * batch_size, len(df_val))

        batch = df_val.iloc[start_idx:end_idx].copy()

        # Calculate music features
        batch = calculate_midi_diff(batch)
        batch = calculate_speed_features(batch)
        batch['black_key'] = batch['midi_number'].apply(is_black_key)
        if 'is_chord' not in batch.columns:
            batch['is_chord'] = 0
        batch['chord'] = batch['is_chord']

        val_batches.append(batch)

        # Free memory
        gc.collect()

    # Concatenate processed batches
    df_val = pd.concat(val_batches, ignore_index=True)
    del val_batches
    gc.collect()

    # Optimize data types for validation data
    for col in float_cols:
        if col in df_val.columns:
            df_val[col] = df_val[col].astype('float32')

    for col in int_cols:
        if col in df_val.columns:
            df_val[col] = df_val[col].astype('int16')

    report_memory()

    # Train Word2Vec model
    print("Training Word2Vec model...")

    # Create word column for training - in batches to save memory
    print("Creating word column for Word2Vec training...")

    # First, ensure all required columns exist
    if not all(col in df_train.columns for col in feature_columns):
        missing_cols = [col for col in feature_columns if col not in df_train.columns]
        print(f"Warning: Missing columns for word creation: {missing_cols}")
        # Use only available columns
        available_cols = [col for col in feature_columns if col in df_train.columns]
        print(f"Using available columns: {available_cols}")
        feature_columns = available_cols

    # First check if 'word' column already exists
    if 'word' not in df_train.columns:
        # Create a new empty 'word' column
        df_train['word'] = ''

    # Check a sample first to see what's going on
    print("\nDEBUG: Inspecting a small sample with verbose output")
    sample_df = df_train.iloc[:100].copy()
    sample_df = create_word_column_verbose(sample_df, feature_columns)

    # Create word column in batches
    word_batch_size = min(50000, len(df_train) // 10)
    for i in tqdm(range(0, len(df_train), word_batch_size)):
        end_idx = min(i + word_batch_size, len(df_train))
        # Process the batch
        batch = df_train.iloc[i:end_idx].copy()
        batch = create_word_column(batch, feature_columns)
        # Assign only the 'word' column back to the original dataframe
        df_train.loc[df_train.index[i:end_idx], 'word'] = batch['word'].values
        gc.collect()

    # Verify the word column exists and has values
    if 'word' not in df_train.columns:
        raise ValueError("Failed to create 'word' column")

    # Check if word column has values
    sample_words = df_train['word'].head(5).tolist()
    print(f"Sample words: {sample_words}")

    # Collect sentences in batches to avoid memory issues
    print("Collecting tokenized sentences for Word2Vec...")
    tokenized_sentences_train = []
    tokenized_sentences_train = []

    for i in tqdm(range(0, len(df_train), batch_size)):
        end_idx = min(i + batch_size, len(df_train))
        batch_sentences = df_train.iloc[i:end_idx]['word'].apply(lambda x: x.split()).tolist()
        tokenized_sentences_train.extend(batch_sentences)

        # Free memory periodically
        if i > 0 and i % (batch_size * 5) == 0:
            gc.collect()

    # Train the model
    print(f"Training Word2Vec on {len(tokenized_sentences_train)} sentences...")
    word2vec_model = train_word2vec(
        tokenized_sentences_train, window=5, vector_size=64, min_count=5, workers=max(1, cpu_count() - 1)
    )
    word2vec_model.save("word2vec_cbow.model")
    print("Word2Vec model saved")

    # Free memory
    del tokenized_sentences_train
    gc.collect()
    report_memory()

    # Process fused features in batches
    print("Processing fused features for training data...")
    batches = []
    scaler_fused = StandardScaler()  # Initialize explicitly

    # Process smaller batches for better memory management
    fused_batch_size = min(10000, len(df_train) // 20)  # Limit batch size
    print(f"Using fused feature batch size: {fused_batch_size}")
    num_batches = (len(df_train) + fused_batch_size - 1) // fused_batch_size

    for i in tqdm(range(num_batches)):
        start_idx = i * fused_batch_size
        end_idx = min((i + 1) * fused_batch_size, len(df_train))

        # Process in even smaller chunks if memory is tight
        if i > 0 and i % 10 == 0:
            print(f"Processing batch {i}/{num_batches}...")
            report_memory()
            gc.collect()

        try:
            batch = df_train.iloc[start_idx:end_idx].copy()

            # Get tokenized sentences for this batch
            batch_sentences = batch['word'].apply(lambda x: x.split()).tolist()

            # Get fused features
            batch = get_fused_features(batch, word2vec_model, batch_sentences)

            # Process fused features
            scaler_fused, fused_features_scaled = process_fused_features(
                batch, scaler_fused=scaler_fused, batch_size=min(1000, len(batch))
            )

            # Store scaled features
            batch['fused_feature_scaled'] = list(fused_features_scaled)

            # Combine features
            batch = combine_features(batch, feature_columns)

            batches.append(batch)
        except Exception as e:
            print(f"Error processing batch {i}: {e}")
            # Skip this batch and continue
            continue

        # Free memory
        gc.collect()

        # Store scaled features
        batch['fused_feature_scaled'] = list(fused_features_scaled)

        # Combine features
        batch = combine_features(batch, feature_columns)

        batches.append(batch)

        # Free memory
        gc.collect()

    # Concatenate processed batches
    df_train_processed = pd.concat(batches, ignore_index=True)

    # Free memory
    del df_train, batches
    gc.collect()
    report_memory()

    # Process validation data similarly
    print("Processing fused features for validation data...")
    batches = []

    num_batches = (len(df_val) + fused_batch_size - 1) // fused_batch_size
    for i in tqdm(range(num_batches)):
        start_idx = i * fused_batch_size
        end_idx = min((i + 1) * fused_batch_size, len(df_val))

        try:
            batch = df_val.iloc[start_idx:end_idx].copy()

            # Create word column if not exists
            if 'word' not in batch.columns:
                batch = create_word_column(batch, feature_columns)

            # Get tokenized sentences for this batch
            batch_sentences = batch['word'].apply(lambda x: x.split()).tolist()

            # Get fused features
            batch = get_fused_features(batch, word2vec_model, batch_sentences)

            # Process fused features (using already fit scaler)
            _, fused_features_scaled = process_fused_features(
                batch, scaler_fused=scaler_fused, batch_size=min(1000, len(batch))
            )

            # Store scaled features
            batch['fused_feature_scaled'] = list(fused_features_scaled)

            # Combine features
            batch = combine_features(batch, feature_columns)

            batches.append(batch)
        except Exception as e:
            print(f"Error processing validation batch {i}: {e}")
            # Skip this batch and continue
            continue

        # Free memory
        gc.collect()

    # Concatenate processed batches
    df_val_processed = pd.concat(batches, ignore_index=True)

    # Free memory
    del df_val, batches
    gc.collect()
    report_memory()

    # Extract features and target variables
    print("Extracting features and targets...")

    # Extract in batches to manage memory
    X_train = []
    y_train = []

    num_batches = (len(df_train_processed) + batch_size - 1) // batch_size
    for i in tqdm(range(num_batches)):
        start_idx = i * batch_size
        end_idx = min((i + 1) * batch_size, len(df_train_processed))

        batch = df_train_processed.iloc[start_idx:end_idx]

        # Extract features and targets
        X_train.append(np.stack(batch['combined_features'].values))
        y_train.append(batch['fingering_encoded'].values)

    # Concatenate batches
    X_train = np.concatenate(X_train)
    y_train = np.concatenate(y_train)

    # Free memory
    del df_train_processed
    gc.collect()
    report_memory()

    # Similarly for validation data
    X_val = []
    y_val = []

    num_batches = (len(df_val_processed) + batch_size - 1) // batch_size
    for i in tqdm(range(num_batches)):
        start_idx = i * batch_size
        end_idx = min((i + 1) * batch_size, len(df_val_processed))

        batch = df_val_processed.iloc[start_idx:end_idx]

        # Extract features and targets
        X_val.append(np.stack(batch['combined_features'].values))
        y_val.append(batch['fingering_encoded'].values)

    # Concatenate batches
    X_val = np.concatenate(X_val)
    y_val = np.concatenate(y_val)

    # Free memory
    del df_val_processed
    gc.collect()
    report_memory()

    # Standardize features
    print("Standardizing features...")

    try:
        print("Attempting to standardize using GPU...")
        # Try GPU acceleration
        X_train_tensor = torch.tensor(X_train, dtype=torch.float32, device='mps')
        X_val_tensor = torch.tensor(X_val, dtype=torch.float32, device='mps')

        # Calculate mean and std
        mean = torch.mean(X_train_tensor, dim=0)
        std = torch.std(X_train_tensor, dim=0)
        std[std == 0] = 1  # Avoid division by zero

        # Standardize
        X_train_scaled = ((X_train_tensor - mean) / std).cpu().numpy()
        X_val_scaled = ((X_val_tensor - mean) / std).cpu().numpy()

        # Free GPU memory
        del X_train_tensor, X_val_tensor
        torch.mps.empty_cache()

        print("GPU standardization successful")
    except Exception as e:
        print(f"GPU standardization failed: {e}")
        print("Using CPU instead...")

        # Use CPU standardization with batches
        scaler = StandardScaler()

        # Fit on training data in batches
        batch_size_scaler = min(100000, len(X_train) // 10)  # Adjust based on available memory
        print(f"Fitting scaler on {len(X_train)} samples in batches of {batch_size_scaler}...")

        for i in tqdm(range(0, len(X_train), batch_size_scaler)):
            end = min(i + batch_size_scaler, len(X_train))
            scaler.partial_fit(X_train[i:end])

        # Transform in batches
        print("Transforming training data...")
        X_train_scaled = np.zeros_like(X_train, dtype=np.float32)
        for i in tqdm(range(0, len(X_train), batch_size_scaler)):
            end = min(i + batch_size_scaler, len(X_train))
            X_train_scaled[i:end] = scaler.transform(X_train[i:end])

        print("Transforming validation data...")
        X_val_scaled = np.zeros_like(X_val, dtype=np.float32)
        for i in tqdm(range(0, len(X_val), batch_size_scaler)):
            end = min(i + batch_size_scaler, len(X_val))
            X_val_scaled[i:end] = scaler.transform(X_val[i:end])

        print("CPU standardization complete")

    # Update variables
    X_train = X_train_scaled
    X_val = X_val_scaled

    # Free memory
    del X_train_scaled, X_val_scaled
    gc.collect()
    report_memory()

    # Generate sequences
    print("Generating sequences...")
    sequence_length = 10
    seq_batch_size = min(10000, len(X_train) // 20)  # No more than 5% at once

    # Train sequences - first save as small chunks
    print("Saving training sequences in small chunks...")
    seq_counter = 0
    X_train_files = []
    y_train_files = []

    for X_batch, y_batch in tqdm(sequence_generator(X_train, y_train, sequence_length, seq_batch_size)):
        x_filename = f'X_train_seq_{seq_counter}.npy'
        y_filename = f'y_train_seq_{seq_counter}.npy'

        np.save(x_filename, X_batch)
        np.save(y_filename, y_batch)

        X_train_files.append(x_filename)
        y_train_files.append(y_filename)
        seq_counter += 1

    # Free memory
    del X_train, y_train
    gc.collect()
    report_memory()

    # Validation sequences
    print("Saving validation sequences in small chunks...")
    seq_counter = 0
    X_val_files = []
    y_val_files = []

    for X_batch, y_batch in tqdm(sequence_generator(X_val, y_val, sequence_length, seq_batch_size)):
        x_filename = f'X_val_seq_{seq_counter}.npy'
        y_filename = f'y_val_seq_{seq_counter}.npy'

        np.save(x_filename, X_batch)
        np.save(y_filename, y_batch)

        X_val_files.append(x_filename)
        y_val_files.append(y_filename)
        seq_counter += 1

    # Free memory
    del X_val, y_val
    gc.collect()
    report_memory()

    # Now combine all chunks into single files if requested
    if not args.keep_chunks:
        try:
            print("Combining all training sequence chunks into single files...")

            # Combine X_train files using memory-mapped approach
            print("Combining X_train chunks...")
            combine_npy_files_safely(X_train_files, 'X_train_combined.npy')

            # Combine y_train files
            print("Combining y_train chunks...")
            combine_npy_files_safely(y_train_files, 'y_train_combined.npy')

            # Combine X_val files
            print("Combining X_val chunks...")
            combine_npy_files_safely(X_val_files, 'X_val_combined.npy')

            # Combine y_val files
            print("Combining y_val chunks...")
            combine_npy_files_safely(y_val_files, 'y_val_combined.npy')

            print("Combination complete! You now have single files for training and validation.")

            # Optionally remove chunk files
            if args.remove_chunks:
                print("Removing chunk files...")
                for file in X_train_files + y_train_files + X_val_files + y_val_files:
                    try:
                        os.remove(file)
                    except Exception as e:
                        print(f"Error removing {file}: {e}")
                print("Chunk files removed.")

        except Exception as e:
            print(f"Error combining chunks: {e}")
            print("Individual chunk files are still available for training.")

    print("Data processing completed successfully!")


if __name__ == "__main__":
    import argparse

    parser = argparse.ArgumentParser(description="Process piano fingering data for model training")
    parser.add_argument("--full", action="store_true", help="Process full dataset instead of sample")
    parser.add_argument("--sample-size", type=int, default=200000, help="Number of samples to use in sample mode")
    parser.add_argument("--keep-chunks", action="store_true", help="Keep data in small chunks without combining")
    parser.add_argument("--remove-chunks", action="store_true", help="Remove small chunk files after combining")

    args = parser.parse_args()

    # Use sample mode by default for testing
    main(sample_mode=not args.full, sample_size=args.sample_size)

,Chord,Note1,Note2,Note3
0,C_maj_2_0,C_2,E_2,G_2
1,C_maj_3_0,C_3,E_3,G_3
2,C_maj_4_0,C_4,E_4,G_4
3,C_maj_5_0,C_5,E_5,G_5
4,C_maj_6_0,C_6,E_6,G_6
5,C_maj_7_0,C_7,E_7,G_7
6,Cs_maj_2_0,Cs_2,F_2,Gs_2
7,Cs_maj_3_0,Cs_3,F_3,Gs_3
8,Cs_maj_4_0,Cs_4,F_4,Gs_4
9,Cs_maj_5_0,Cs_5,F_5,Gs_5


In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import DataLoader, Dataset
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm
import pickle
from data_utils import load_pickle  # 用于加载 LabelEncoder
from models import BiLSTMWithAttention  # 示例模型，可以替换为其他模型
from torch.cuda.amp import GradScaler, autocast


# Focal Loss 定义，用于处理类别不平衡和难分类样本
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean', ignore_index=10):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
        self.ignore_index = ignore_index

    def forward(self, inputs, targets):
        mask = targets != self.ignore_index
        inputs = inputs[mask]
        targets = targets[mask]
        if inputs.size(0) == 0:
            return torch.tensor(0.0, device=inputs.device)

        BCE_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-BCE_loss)
        F_loss = self.alpha * (1 - pt) ** self.gamma * BCE_loss

        if self.reduction == 'mean':
            return F_loss.mean()
        elif self.reduction == 'sum':
            return F_loss.sum()
        else:
            return F_loss


# 自定义 Dataset，支持动态镜像增强
class FingeringDataset(Dataset):
    def __init__(self, X, y, le_hand, hand_index=2, mirror_prob=0.5):
        self.X = X
        self.y = y
        self.le_hand = le_hand
        self.hand_index = hand_index
        self.mirror_prob = mirror_prob

        # 打印形状信息以便调试
        print(f"X shape: {self.X.shape}, y shape: {self.y.shape}")
        # 输出数据类型信息
        print(f"X dtype: {self.X.dtype}, y dtype: {self.y.dtype}")
        # 输出一个样本示例
        print(f"Sample X[0] shape: {self.X[0].shape}, y[0]: {self.y[0]}")

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        X_sample = self.X[idx].copy()
        y_sample = int(self.y[idx])  # 将y_sample转换为整数，因为它是一个单一的标签值

        # 动态镜像增强
        if np.random.rand() < self.mirror_prob:
            # 检查最后一个时间步的手属性
            if X_sample[-1, self.hand_index] == self.le_hand.transform(['left'])[0]:
                # 将所有时间步的手属性改为右手
                X_sample[:, self.hand_index] = self.le_hand.transform(['right'])[0]
            else:
                # 将所有时间步的手属性改为左手
                X_sample[:, self.hand_index] = self.le_hand.transform(['left'])[0]

            # 指法翻转规则 - 直接应用到单一的标签值
            finger_flip = {0: 4, 1: 3, 2: 2, 3: 1, 4: 0, 5: 9, 6: 8, 7: 7, 8: 6, 9: 5}
            y_sample = finger_flip.get(y_sample, y_sample)

        # 返回转换为张量的样本
        return torch.tensor(X_sample, dtype=torch.float32), torch.tensor(y_sample, dtype=torch.long)


def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    # scaler = GradScaler()
    # # 在 train_epoch 中：
    # with autocast():
    #     outputs = model(X_batch)
    #     loss = criterion(outputs, y_batch)
    # scaler.scale(loss).backward()
    # scaler.step(optimizer)
    # scaler.update()
    for X_batch, y_batch in tqdm(loader, desc="Training"):
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # 梯度裁剪
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += (y_batch != 10).sum().item()
            correct += (predicted == y_batch).sum().item()
    avg_loss = total_loss / len(loader)
    accuracy = correct / total if total > 0 else 0
    return avg_loss, accuracy


def main():
    # 数据路径
    train_X_path = 'X_train_combined.npy'
    train_y_path = 'y_train_combined.npy'
    val_X_path = 'X_val_combined.npy'
    val_y_path = 'y_val_combined.npy'

    # 打印数据信息
    print(f"加载训练和验证数据...")

    # 加载数据
    X_train = np.load(train_X_path, mmap_mode='r')
    y_train = np.load(train_y_path, mmap_mode='r')
    X_val = np.load(val_X_path, mmap_mode='r')
    y_val = np.load(val_y_path, mmap_mode='r')

    print(f"数据加载完成。X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
    print(f"X_val shape: {X_val.shape}, y_val shape: {y_val.shape}")

    # 加载 LabelEncoders
    le_hand = load_pickle('le_hand.pkl')
    le_fingering = load_pickle('le_fingering.pkl')

    # 分离标注和未标注数据（未标注数据标记为10）
    labeled_idx = y_train != 10
    X_train_labeled = X_train[labeled_idx]
    y_train_labeled = y_train[labeled_idx]
    X_train_unlabeled = X_train[~labeled_idx]

    print(f"已标注数据: {len(X_train_labeled)}, 未标注数据: {len(X_train_unlabeled)}")

    # 模型参数
    input_size = X_train.shape[2]
    hidden_size = 256
    num_layers = 2
    num_classes = 10
    dropout = 0.3

    print(f"模型参数 - input_size: {input_size}, hidden_size: {hidden_size}, num_classes: {num_classes}")

    # 初始化模型
    model = BiLSTMWithAttention(input_size, hidden_size, num_layers, num_classes, dropout)
    # device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    model.to(device)
    print(f"使用设备: {device}")

    # 数据加载器
    train_dataset = FingeringDataset(X_train_labeled, y_train_labeled, le_hand, mirror_prob=0.5)
    val_dataset = FingeringDataset(X_val, y_val, le_hand, mirror_prob=0.0)  # 验证集不增强
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

    # 优化器、损失函数和调度器
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
    criterion = FocalLoss(alpha=1, gamma=2, ignore_index=10)
    scheduler = CosineAnnealingLR(optimizer, T_max=10, eta_min=0.0001)

    # 训练参数
    num_epochs = 50
    best_val_loss = float('inf')
    patience = 5
    trigger_times = 0

    print("开始初始训练...")
    for epoch in range(num_epochs):
        train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_accuracy = evaluate(model, val_loader, criterion, device)

        print(f"Epoch [{epoch + 1}/{num_epochs}], Train Loss: {train_loss:.4f}, "
              f"Val Loss: {val_loss:.4f}, Val Acc: {val_accuracy:.4f}")

        scheduler.step()

        # 早停机制
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            trigger_times = 0
            torch.save(model.state_dict(), 'best_model.pth')
        else:
            trigger_times += 1
            if trigger_times >= patience:
                print("触发早停机制，停止训练！")
                break

    # 自训练阶段
    if len(X_train_unlabeled) > 0:
        print("开始自训练...")
        model.eval()

        # 处理未标注数据的批次大小
        batch_size = 256
        all_pseudo_X = []
        all_pseudo_y = []

        # 分批处理未标注数据以避免内存问题
        for i in range(0, len(X_train_unlabeled), batch_size):
            end = min(i + batch_size, len(X_train_unlabeled))
            batch_X = X_train_unlabeled[i:end]

            with torch.no_grad():
                X_unlabeled_tensor = torch.tensor(batch_X, dtype=torch.float32).to(device)
                outputs = model(X_unlabeled_tensor)
                probabilities, predicted = torch.max(F.softmax(outputs, dim=1), 1)
                high_conf_idx = probabilities > 0.9  # 置信度阈值

                # 收集高置信度的样本
                if high_conf_idx.sum().item() > 0:
                    pseudo_X = batch_X[high_conf_idx.cpu().numpy()]
                    pseudo_y = predicted[high_conf_idx].cpu().numpy()

                    all_pseudo_X.append(pseudo_X)
                    all_pseudo_y.append(pseudo_y)

        # 合并所有伪标签样本
        if all_pseudo_X:
            pseudo_X = np.concatenate(all_pseudo_X, axis=0)
            pseudo_y = np.concatenate(all_pseudo_y, axis=0)

            print(f"添加 {len(pseudo_X)} 个伪标签样本到训练集")
            X_train_new = np.concatenate([X_train_labeled, pseudo_X], axis=0)
            y_train_new = np.concatenate([y_train_labeled, pseudo_y], axis=0)
            train_dataset = FingeringDataset(X_train_new, y_train_new, le_hand, mirror_prob=0.5)
            train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

            # 继续训练
            model.load_state_dict(torch.load('best_model.pth'))  # 加载最佳模型
            for epoch in range(10):  # 自训练10个额外epoch
                train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
                val_loss, val_accuracy = evaluate(model, val_loader, criterion, device)
                print(f"Self-Training Epoch [{epoch + 1}/10], Train Loss: {train_loss:.4f}, "
                      f"Val Loss: {val_loss:.4f}, Val Acc: {val_accuracy:.4f}")
                scheduler.step()

    # 保存最终模型
    torch.save(model.state_dict(), 'fingering_model_final.pth')
    print("训练完成，模型已保存至 'fingering_model_final.pth'")


if __name__ == "__main__":
    main()